# Plotly交互可视化

学习目标：绘制散点、折线、气泡、饼图、环形图、层级图和桑基图，逐步调整颜色、标签与悬停，并保存交互 HTML。

前置知识：Python 字典与循环、模块导入、pandas 表格选择与排序、横纵坐标和图例。

运行环境：Python 3.12、Plotly 7.1、pandas 3.0；浏览器用于查看导出结果。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章所有读数均为教学模拟。Notebook 使用 plotly_mimetype 输出交互图，适用于支持该格式的 VS Code Notebook、JupyterLab 等前端；阅读器未呈现图形时，可打开第 12 节导出的 HTML。第 13 节静态导出为选学，默认关闭。

阅读安排：每个图表单元先准备数据，再创建图形、调整参数并显示结果。第 12 节组合与导出作为补充练习。

静态阅读：组合示例附近附有实际页面截图；运行本章或打开导出的 HTML 可操作交互。

## 1 从一张小表画散点图

In [1]:
# 适用场景：散点图查看加热功率与温度的关系；每个点是一条观测。
import pandas as pd
import plotly.express as px
import plotly.io as pio

# plotly_mimetype 由 Notebook 前端绘制；未显示时可打开第 12 节的 HTML。
pio.renderers.default = "plotly_mimetype"

# 数据组织：每行一组教学模拟读数，功率单位 W，温度单位 °C。
readings = pd.DataFrame(
    {"power_w": [10, 15, 20, 25], "temperature_c": [22.0, 24.0, 25.5, 28.0]}
)

# 最小绘制代码：Plotly Express（px）接收表格和列名，返回 Figure。
fig = px.scatter(readings, x="power_w", y="temperature_c")

# 常用参数调整：update_layout 设置整张图的外观；轴标题补充单位。
# template="plotly_white" 使用白底；删去以下调整也能画出散点。
fig.update_layout(
    template="plotly_white",
    xaxis_title="加热功率（W）",
    yaxis_title="温度（°C）",
)

# 实际成图：show() 显示 4 个点；悬停 20 W 的点，读数应为 25.5 °C。
fig.show()

## 2 用字段映射颜色和点形

In [2]:
# 适用场景：分组散点图比较不同方案的观测，用颜色和点形区分组别。
# 数据组织：一行是某方案某时刻的一次模拟读数；sample 为编号。
# minute 为分钟、temperature_c 为 °C、power_w 为 W；没有重复测量或聚合。
measurements = pd.DataFrame(
    {
        "sample": ["A0", "A2", "A4", "A6", "A8", "B0", "B2", "B4", "B6", "B8"],
        "scheme": ["A"] * 5 + ["B"] * 5,
        "minute": [0, 2, 4, 6, 8] * 2,
        "temperature_c": [21.0, 23.5, 27.0, 29.0, 30.0, 21.0, 22.5, 24.0, 25.5, 26.0],
        "power_w": [30] * 5 + [20] * 5,
    }
)
# 常用标签：labels 将字段名替换为中文名称与单位，后续图形也会复用。
labels = {
    "minute": "时间（min）",
    "temperature_c": "温度（°C）",
    "scheme": "方案",
    "power_w": "功率（W）",
}
colors = {"A": "#0072B2", "B": "#D55E00"}

# 最小绘制代码：在 x、y 之外加入 color="scheme"，即可按方案分组。
# 常用参数调整：symbol 再用点形分组；两份 map 固定颜色与点形，
# category_orders 固定图例顺序。scheme 是类别；数值字段通常会使用连续色标。
group_fig = px.scatter(
    measurements,
    x="minute",
    y="temperature_c",
    color="scheme",
    symbol="scheme",
    color_discrete_map=colors,
    symbol_map={"A": "circle", "B": "square"},
    category_orders={"scheme": ["A", "B"]},
    labels=labels,
    template="plotly_white",
)
print(
    measurements[["sample", "scheme", "minute", "temperature_c"]].to_string(index=False)
)

# 实际成图：10 条观测中 A、B 各有 5 个点，起点重合、后续温度不同。
# A 为蓝色圆点、B 为橙色方点。
group_fig.show()

sample scheme  minute  temperature_c
    A0      A       0           21.0
    A2      A       2           23.5
    A4      A       4           27.0
    A6      A       6           29.0
    A8      A       8           30.0
    B0      B       0           21.0
    B2      B       2           22.5
    B4      B       4           24.0
    B6      B       6           25.5
    B8      B       8           26.0


## 3 Figure 的结构与定制

### 3.1 data、layout 和 trace

Plotly 的 Figure 保存图形定义；它与其他绘图库中同名的对象并不是同一个类。一个 Figure 的 data 包含若干 trace，每个 trace 是一组按相同方式绘制的数据，例如同一方案的一组散点。

| 名称 | 中文名称／含义 | 本例保存的内容 |
| --- | --- | --- |
| Figure | 完整图形定义 | 数据系列、布局及可选动画帧 |
| data | 数据系列集合 | 两组 scatter trace |
| trace | 一条数据系列 | 一组横纵坐标、名称和点样式 |
| layout | 布局设置 | 标题、坐标轴、图例、字体和尺寸 |
| frames | 动画帧集合 | 本例未使用，为空 |

本例按方案映射颜色与点形，因此产生两个 trace，不是每行数据产生一个 trace。先查看结构，再决定应该修改某个数据系列还是整张图。

In [3]:
print(type(group_fig).__name__, len(group_fig.data))  # 预期：Figure 2。
for trace in group_fig.data:
    print(trace.name, trace.type, len(trace.x), trace.marker.color)
# 预期：A scatter 5 #0072B2；B scatter 5 #D55E00。
print(group_fig.layout.xaxis.title.text)  # 预期：时间（min）。
print(len(group_fig.frames))  # 预期：0；本图没有动画帧。

Figure 2
A scatter 5 #0072B2
B scatter 5 #D55E00
时间（min）
0


### 3.2 分别更新数据样式与整体布局

update_traces() 修改数据系列属性；不指定 selector 时会更新所有系列，指定 selector={"name": "A"} 时只匹配名称为 A 的系列。update_layout() 修改图的整体设置，update_xaxes() 和 update_yaxes() 用于坐标轴。

marker_size 是嵌套属性 marker.size 的简写。下面先把两组点设为同样大小，再通过 selector 给 A 的点增加轮廓。height 的单位是像素；字体按列表由浏览器选择，本机使用 Microsoft YaHei，其他系统应换成已安装且支持中文的字体。

这些 update 方法会修改现有 Figure。需要保留另一种方案时，可以重新创建图，避免把多次试验的状态混在一起。

In [4]:
group_fig.update_traces(marker_size=10)
group_fig.update_traces(
    marker_line={"width": 1, "color": "#333333"}, selector={"name": "A"}
)
group_fig.update_layout(
    title={"text": "两种加热方案的模拟读数", "x": 0.05},
    font={"family": "Microsoft YaHei, Arial, sans-serif", "size": 14},
    height=420,
    legend={"title_text": "方案"},
)
group_fig.update_xaxes(range=[-0.4, 8.4], dtick=2)
group_fig.update_yaxes(range=[20, 32], dtick=2)
group_fig.show()  # A 为带深色轮廓的圆点，B 为方点；文字与单位应完整。

## 4 悬停内容与数字格式

### 4.1 用 hover_name 和 hover_data 选择字段

hover_name 指定悬停提示的加粗标题，hover_data 选择额外显示的字段，也可用字典控制格式。字典值 True 表示显示，False 表示不显示；数值格式 ":.1f" 表示保留一位小数。

下面将记录编号放在提示顶部，额外显示功率。坐标轴上没有功率这一维，悬停使读者能逐点补充查看，而不必把所有信息都挤进点旁文字。

In [5]:
hover_fig = px.scatter(
    measurements,
    x="minute",
    y="temperature_c",
    color="scheme",
    color_discrete_map=colors,
    hover_name="sample",
    hover_data={"power_w": True, "temperature_c": ":.1f", "scheme": False},
    labels=labels,
    template="plotly_white",
)
hover_fig.update_traces(marker_size=11)
hover_fig.update_layout(height=400, font_family="Microsoft YaHei, Arial, sans-serif")
hover_fig.show()  # 悬停 A4：标题 A4、时间 4 min、温度 27.0 °C、功率 30 W。

### 4.2 用 hovertemplate 指定完整提示

需要精确安排字段顺序时，先用 custom_data 给每个点附加字段，再通过 update_traces 设置 hovertemplate。

%{x} 和 %{y} 分别取当前点的横纵坐标；%{customdata[0]} 取 custom_data 列表的第一个字段，索引从 0 开始。下面列表顺序为 sample、scheme、power_w，因此索引 0、1、2 分别对应记录编号、方案和功率。

模板里的 %{y:.1f} 给温度保留一位小数，&lt;br&gt; 换行，&lt;b&gt; 标记加粗文字；&lt;extra&gt;&lt;/extra&gt; 隐藏附加的系列名框。这里使用普通字符串，替换由 Plotly 完成，不是 Python 的 f-string。模板变量名中的 customdata 没有下划线，与 px 参数 custom_data 的拼写不同。

In [6]:
# custom_data 按 sample、scheme、power_w 顺序附在每个点上，供悬停模板取值。
template_fig = px.scatter(
    measurements,
    x="minute",
    y="temperature_c",
    color="scheme",
    color_discrete_map=colors,
    custom_data=["sample", "scheme", "power_w"],
    labels=labels,
    template="plotly_white",
)
# customdata 的索引对应上面字段顺序；x、y 直接取点坐标。
# .0f、.1f 控制小数位，空 extra 隐藏额外的系列名称框。
hover_template = (
    "<b>记录 %{customdata[0]}</b><br>"
    "方案：%{customdata[1]}<br>"
    "时间：%{x:.0f} min<br>"
    "温度：%{y:.1f} °C<br>"
    "功率：%{customdata[2]} W"
    "<extra></extra>"
)
template_fig.update_traces(marker_size=11, hovertemplate=hover_template)
template_fig.update_layout(height=400, font_family="Microsoft YaHei, Arial, sans-serif")
template_fig.show()  # A4 的提示包含编号、方案 A、4 min、27.0 °C 和 30 W。

## 5 有顺序的数据再连接成线

In [7]:
# 适用场景：折线图表达各方案随时间的变化，独立且无顺序的样本不宜连线。
# 数据组织：复用 measurements、labels、colors；每行仍是一次读数。
# px.line 按输入顺序连接，不会自动按横坐标排序，因此先按方案和时间排序。
ordered = measurements.sort_values(["scheme", "minute"])
# 最小绘制代码：x、y 给出时间与温度，color 将两个方案分为两条线。
# 常用参数调整：markers=True 保留观测点；line_dash 与 map 固定实线、虚线。
line_fig = px.line(
    ordered,
    x="minute",
    y="temperature_c",
    color="scheme",
    line_dash="scheme",
    color_discrete_map=colors,
    line_dash_map={"A": "solid", "B": "dash"},
    markers=True,
    labels=labels,
    template="plotly_white",
)
# x unified 合并同一横坐标附近的系列提示；不能逐一展示同系列同一 x 的重复值。
# height 为像素；浏览器按字体列表选择，其他系统可换成本机中文字体。
line_fig.update_layout(
    height=420, hovermode="x unified", font_family="Microsoft YaHei, Arial, sans-serif"
)

# 实际成图：连线连接已有读数，不代表中间时刻也经过测量。
# 悬停 4 min，可并列查看 A 的 27.0 °C 和 B 的 24.0 °C。
line_fig.show()

## 6 缩放、平移与图例操作

layout 的 dragmode 决定拖动绘图区的动作；"zoom" 用矩形框选择要放大的范围，"pan" 用于平移。显示时的 config 控制工具栏和输入行为，它不属于 layout；导出时需要再传同一份 config，才能保留相同配置。

本例启用 scrollZoom，鼠标滚轮也可缩放。doubleClick="reset" 使双击绘图区恢复初始范围；这与双击图例的行为不同。图例的 itemclick="toggle" 切换一个系列的可见性，itemdoubleclick="toggleothers" 用于突出所选系列。

在图中依次尝试：悬停读数，拖框放大 2～6 分钟，切换工具栏的平移按钮，双击绘图区复原；单击 B 图例隐藏 B，再次单击显示 B。隐藏只是交互显示状态，不能据此把原数据的组别或观测数改写掉。

In [8]:
interaction_config = {
    "scrollZoom": True,
    "displayModeBar": True,
    "displaylogo": False,
    "responsive": True,
    "doubleClick": "reset",
}
line_fig.update_layout(
    dragmode="zoom",
    legend={"itemclick": "toggle", "itemdoubleclick": "toggleothers"},
)
line_fig.update_xaxes(range=[-0.4, 8.4], dtick=2)
line_fig.update_yaxes(range=[20, 32], dtick=2)
line_fig.show(config=interaction_config)
# 放大后刻度范围应缩小；双击绘图区恢复初始范围。
# 单击 B 图例后只见蓝色 A，再单击 B 恢复橙色虚线。

## 7 气泡图：用大小增加一个数量维度

In [9]:
# 适用场景：气泡图在两个位置变量之外，用圆的大小表达第三个数量变量。
# 数据组织：一行一个模拟服务站；distance_km 为距离，delivery_min 为平均时间，
# orders 为同一天的正订单数；大小编码不能表达正负方向。
stations = pd.DataFrame(
    {
        "station": ["站点 A", "站点 B", "站点 C", "站点 D"],
        "distance_km": [2, 4, 6, 8],
        "delivery_min": [12, 18, 16, 25],
        "orders": [20, 40, 80, 60],
    }
)
# 最小绘制代码：在散点图的 x、y 之外增加 size="orders"。
# 常用参数调整：size_max 控制最大标记大小；hover_name 显示站名，
# hover_data 保留精确订单数，labels 补充单位。
bubble_fig = px.scatter(
    stations,
    x="distance_km",
    y="delivery_min",
    size="orders",
    size_max=48,
    hover_name="station",
    hover_data={"orders": True},
    labels={
        "distance_km": "距仓库距离（km）",
        "delivery_min": "平均配送时间（min）",
        "orders": "订单数（单）",
    },
    title="服务站距离、时间与订单数（教学模拟）",
    template="plotly_white",
)
# sizemode="area" 让面积随数量变化，不能把直径直接读成订单数。
bubble_fig.update_traces(marker_sizemode="area", marker_color="#0072B2")
bubble_fig.update_layout(height=440, font_family="Microsoft YaHei, Arial, sans-serif")

# 实际成图：站点 C 的圆最大，悬停读到 6 km、16 min、80 单。
# D 的时间为 25 min，圆却比 C 小；大小和纵坐标表达不同数量。
# 显示时复用 interaction_config 的缩放与工具栏设置。
bubble_fig.show(config=interaction_config)

## 8 饼图与环形图：看同一整体的组成

### 8.1 把类别数量转换为扇区

In [10]:
# 适用场景：饼图表达同一整体中互不重复的类别占比。
# 数据组织：一行一个渠道，orders 为模拟订单数；计数非负、整体大于 0。
# 这里三类互斥且完整，共 100 单；漏掉类别会改变百分比的分母。
# names 标签重复时 Plotly 会合并扇区，输入重复行是否应累计要先核对。
channels = pd.DataFrame({"channel": ["直营", "代理", "线上"], "orders": [50, 30, 20]})
channel_colors = {"直营": "#0072B2", "代理": "#D55E00", "线上": "#009E73"}
pie_hover = "%{label}<br>%{value} 单<br>当前图中占比：%{percent:.0%}<extra></extra>"
# 最小绘制代码：names 给类别，values 给数量。
# 常用参数调整：color 与 map 固定各渠道颜色；标题保留整体总数。
pie_fig = px.pie(
    channels,
    names="channel",
    values="orders",
    color="channel",
    color_discrete_map=channel_colors,
    title="一天的订单渠道组成（教学模拟，共 100 单）",
    template="plotly_white",
)
# textinfo 显示标签和百分比，sort=False 保留输入顺序。
# 提示中的 label/value/percent 分别为类别/数量/占比，.0% 显示整数百分数。
pie_fig.update_traces(textinfo="label+percent", sort=False, hovertemplate=pie_hover)
pie_fig.update_layout(height=440, font_family="Microsoft YaHei, Arial, sans-serif")
print("原始订单总数：", channels["orders"].sum())  # 100 单。

# 实际成图：直营 50%、代理 30%、线上 20%，悬停能读到原始数量。
# 单击图例可隐藏渠道；比较占比前先恢复相同的显示类别。
pie_fig.show(config=interaction_config)

原始订单总数： 100


### 8.2 用 hole 留出圆心

In [11]:
# 适用场景：环形图同样表达整体组成，圆心留白不改变数据含义。
# 数据组织：复用 channels、channel_colors 和 pie_hover，仍为同一天的 100 单。
# 最小绘制代码：px.pie 增加 hole 即可生成圆环。
# 常用参数调整：hole 为内圆半径与原半径之比，范围 0～1；这里设为 0.45。
# labels、values 与颜色不变，外径不用于比较不同日期的订单总量。
donut_fig = px.pie(
    channels,
    names="channel",
    values="orders",
    color="channel",
    color_discrete_map=channel_colors,
    hole=0.45,
    title="订单渠道环形图（教学模拟，共 100 单）",
    template="plotly_white",
)
donut_fig.update_traces(textinfo="label+percent", sort=False, hovertemplate=pie_hover)
donut_fig.update_layout(height=440, font_family="Microsoft YaHei, Arial, sans-serif")

# 实际成图：仍为 50%、30%、20%，圆心留白不表示缺少了 45% 的订单。
# 隐藏“线上”再恢复，观察显示占比变化；原始表格仍是 100 单。
donut_fig.show(config=interaction_config)

## 9 进阶：矩形树图表示父子组成

In [12]:
# 适用场景：矩形树图（treemap）用嵌套矩形表达层级，用面积表达数量。
# 数据组织：每行是一个地区下某产品的模拟订单数，四行互不重复，共 100 单。
# 父级由叶子汇总，不要再把北区 60、南区 40 作为额外叶子加入同一表。
region_orders = pd.DataFrame(
    {
        "region": ["北区", "北区", "南区", "南区"],
        "product": ["设备", "配件", "设备", "配件"],
        "orders": [40, 20, 10, 30],
    }
)
region_colors = {"北区": "#0072B2", "南区": "#D55E00", "(?)": "#EEEEEE"}
# 最小绘制代码：path 从根到叶列出层级，values 决定面积。
# px.Constant 增加共同根节点；常用参数 color 只区分地区。
treemap_fig = px.treemap(
    region_orders,
    path=[px.Constant("全部订单"), "region", "product"],
    values="orders",
    color="region",
    color_discrete_map=region_colors,
    title="地区与产品订单层级（教学模拟）",
    template="plotly_white",
)
# 常用参数调整：root_color 设置根颜色；textinfo 显示名称与数量。
# hovertemplate 保留精确订单数，避免仅凭面积判断。
treemap_fig.update_traces(
    root_color="#EEEEEE",
    textinfo="label+value",
    hovertemplate="%{label}<br>订单数：%{value} 单<extra></extra>",
)
treemap_fig.update_layout(height=460, font_family="Microsoft YaHei, Arial, sans-serif")
print(region_orders.groupby("region")["orders"].sum())  # 北区 60，南区 40。

# 实际成图：点击北区，设备仍为 40、配件仍为 20 单；放大不表示数量增加。
# 点击路径栏“全部订单”返回上层。
treemap_fig.show(config=interaction_config)

region
北区    60
南区    40
Name: orders, dtype: int64


## 10 进阶：旭日图中的父级总量

In [13]:
# 适用场景：旭日图（sunburst）用由内向外的圆环表达父子层级和数量分配。
# 数据组织：仍是上节的 100 单，这次显式给出每个节点及其父级。
# ids 唯一；parents 引用父节点 id，空字符串表示无父级；同名“设备”有不同 id。
# branchvalues="total" 表示父节点值包含子级，子级之和不能超过父级。
# 本例 100=60+40、60=40+20、40=10+30；各层不能重复累加成订单总数。
# "remainder" 表示父级值是子级之外的额外量，不能沿用本数据直接切换。
import plotly.graph_objects as go

# 最小绘制代码：go.Sunburst 接收 ids、labels、parents、values，
# go.Figure 将数据系列包装为完整图形。
sunburst_fig = go.Figure(
    go.Sunburst(
        ids=["all", "north", "south", "n-device", "n-part", "s-device", "s-part"],
        labels=["全部订单", "北区", "南区", "设备", "配件", "设备", "配件"],
        parents=["", "all", "all", "north", "north", "south", "south"],
        values=[100, 60, 40, 40, 20, 10, 30],
        branchvalues="total",
        # 常用参数调整：marker_colors 固定各节点颜色，textinfo 显示名称与数量。
        marker_colors=[
            "#EEEEEE",
            "#0072B2",
            "#D55E00",
            "#0072B2",
            "#0072B2",
            "#D55E00",
            "#D55E00",
        ],
        textinfo="label+value",
        # percentParent 分母是直接父级，percentRoot 分母是根节点。
        hovertemplate=(
            "%{label}<br>订单数：%{value} 单<br>"
            "占父级：%{percentParent:.1%}<br>"
            "占全部：%{percentRoot:.1%}<extra></extra>"
        ),
    )
)
sunburst_fig.update_layout(
    title="同一组订单的旭日图（教学模拟，共 100 单）",
    template="plotly_white",
    height=480,
    font_family="Microsoft YaHei, Arial, sans-serif",
)

# 实际成图：北区设备 40 单，占北区约 66.7%、占全部 40.0%。
# 点击北区展开，再点击中心返回；不能把层级重复计为独立订单。
sunburst_fig.show(config=interaction_config)

## 11 进阶：桑基图表示流向与数量

In [14]:
# 适用场景：桑基图（Sankey diagram）表达阶段之间的去向，连线宽度表示流量。
# 数据组织：同一批 100 件模拟工单先分配，再转为已解决或升级处理。
# source/target 为从 0 开始的节点索引，三个 flow 列表按位置一一对应。
# 节点位置不是连续时间轴；真实数据若有新增或退出，需明确列出相应去向。
node_labels = ["收到工单", "自动处理", "人工处理", "已解决", "升级处理"]
flow_sources = [0, 0, 1, 1, 2, 2]
flow_targets = [1, 2, 3, 4, 3, 4]
flow_values = [60, 40, 50, 10, 30, 10]

# 最小绘制代码：node.label 给节点名称，link 给起点、终点和数量。
sankey_fig = go.Figure(
    go.Sankey(
        # 常用参数调整：valueformat 控制整数显示，valuesuffix 补上“件”。
        valueformat=".0f",
        valuesuffix=" 件",
        node={
            "label": node_labels,
            "color": ["#666666", "#0072B2", "#D55E00", "#009E73", "#CC79A7"],
            # pad 为节点间距，thickness 为节点宽度，两者单位为像素。
            "pad": 20,
            "thickness": 20,
        },
        link={"source": flow_sources, "target": flow_targets, "value": flow_values},
    )
)
sankey_fig.update_layout(
    title="100 件工单的两阶段流向（教学模拟）",
    template="plotly_white",
    height=440,
    font={"family": "Microsoft YaHei, Arial, sans-serif", "size": 14},
)
print("第一阶段：", sum(flow_values[:2]), "件")  # 100 件。
print("第二阶段：", sum(flow_values[2:]), "件")  # 同一批 100 件。

# 实际成图：悬停“自动处理 → 已解决”读到 50 件，流入“已解决”共 80 件。
# 同一工单经过两个阶段，所有连接之和为 200，实际工单仍只有 100 件。
# 自动处理流入 60、流出 50+10；悬停时同时核对连接两端的名称。
sankey_fig.show(config=interaction_config)

第一阶段： 100 件
第二阶段： 100 件


## 12 补充练习：组合设置与 HTML 导出

### 12.1 把配色、线型和悬停放在同一张图中

静态预览来自本节导出页面：鼠标位于 A4，提示显示 4 min、27.0 °C 和 30 W。运行图形或打开 HTML 后可自行悬停、缩放和切换图例。

![Plotly 组合示例：两组温度曲线及 A4 的时间、温度和功率提示](image/03-plotly-hover.png)

In [15]:
# 补充练习：复用 ordered、colors、labels、hover_template 和 interaction_config。
# 每个时间点仍是单次模拟读数，两方案功率不同，温度高低不能直接说明效率或因果。
# hovermode="closest" 逐点显示完整提示；margin 的 l/r/t/b 是左右上下像素边距。
report_fig = px.line(
    ordered,
    x="minute",
    y="temperature_c",
    color="scheme",
    line_dash="scheme",
    color_discrete_map=colors,
    line_dash_map={"A": "solid", "B": "dash"},
    category_orders={"scheme": ["A", "B"]},
    custom_data=["sample", "scheme", "power_w"],
    markers=True,
    labels=labels,
    template="plotly_white",
)
# 先统一系列线条与悬停内容，再调整整张图的标题、图例与交互。
report_fig.update_traces(marker_size=9, line_width=2, hovertemplate=hover_template)
report_fig.update_layout(
    title={
        "text": "两种加热方案的教学模拟<br><sup>A：30 W；B：20 W；各时刻为单次读数</sup>",
        "x": 0.05,
    },
    font={"family": "Microsoft YaHei, Arial, sans-serif", "size": 14},
    height=480,
    margin={"l": 70, "r": 50, "t": 100, "b": 65},
    hovermode="closest",
    dragmode="zoom",
    legend={
        "title_text": "方案",
        "itemclick": "toggle",
        "itemdoubleclick": "toggleothers",
    },
)
report_fig.update_xaxes(range=[-0.4, 8.4], dtick=2, gridcolor="#EEEEEE")
report_fig.update_yaxes(range=[20, 32], dtick=2, gridcolor="#EEEEEE")
report_fig.show(config=interaction_config)
# 共 2 个系列、10 个数据点；A4 对应 (4, 27)，B4 对应 (4, 24)。
# 尝试把 marker_size 从 9 改为 12，观察点与折线的层次。

### 12.2 内嵌 JavaScript 与 CDN

write_html 保存浏览器可以打开的图形页面。full_html=True 输出完整 HTML；include_plotlyjs 决定页面如何取得绘图所需的 Plotly.js，不能把“已生成 HTML 文件”直接等同于“任何环境都能离线打开”。

| include_plotlyjs 的值 | 中文名称／含义 | 本章的使用条件 |
| --- | --- | --- |
| True | 将 Plotly.js 内嵌到 HTML | 文件较大，本例数据与脚本都随页面保存 |
| "cdn" | 从外部内容分发网络加载 Plotly.js | 文件较小，首次加载脚本需要网络；浏览器缓存不等于稳定离线支持 |
| False | 不附带 Plotly.js | 只适用于外层页面已加载对应脚本的嵌入场景，不能把它单独当成完整可运行页面 |

本例没有远程图片、在线地图、外部字体文件或数学公式脚本。True 版因此具备独立离线使用所需的绘图资源；若后续增加其他外部资源，还要分别处理。CDN 版引用的 Plotly.js URL 与包内版本匹配，数据本身仍写在 HTML 中。

两种导出都在浏览器内处理悬停、缩放和图例，无需持续运行 Python 服务。它们保存的是导出时的 Figure 定义，不应把浏览器中临时隐藏图例或缩放后的视图当成已保存到 Python Figure。

In [16]:
import tempfile
from pathlib import Path

# 同一张图导出两份页面，只改变 Plotly.js 是写入文件还是从 CDN 加载。
output_dir = Path(tempfile.mkdtemp(prefix="plotly-ch03-"))
inline_path = output_dir / "plotly-inline.html"
cdn_path = output_dir / "plotly-cdn.html"

report_fig.write_html(
    inline_path,
    include_plotlyjs=True,
    full_html=True,
    config=interaction_config,
    auto_open=False,
)
report_fig.write_html(
    cdn_path,
    include_plotlyjs="cdn",
    full_html=True,
    config=interaction_config,
    auto_open=False,
)
# 保留目录供浏览器查看，完成后按本章清理单元删除。
print("输出目录：", output_dir)
print("内嵌版大小：", inline_path.stat().st_size, "字节")
print("CDN 版大小：", cdn_path.stat().st_size, "字节")
# 内嵌版通常明显更大，因为包含 Plotly.js；具体字节数取决于版本和数据。

输出目录： C:\Users\ZHUANG\AppData\Local\Temp\plotly-ch03-c_fkjjsq
内嵌版大小： 4829216 字节
CDN 版大小： 9933 字节


### 12.3 为不同图形各保存一份页面

把第 7～11 节的六张图分别保存为独立 HTML，统一使用 include_plotlyjs=True。字典将文件名与 Figure 对应；每次调用 write_html 都把图形数据和脚本放入该文件，因此文件可以分别分享。as_uri() 把文件的绝对路径转换为浏览器使用的 file 地址。本例没有额外远程数据或脚本依赖。

保留各图自己的标题与提示，导出时继续传入 interaction_config。树图、旭日图和桑基图不使用普通散点图的横纵坐标缩放方式；打开后可尝试各自的悬停、图例或层级交互。

In [17]:
# 文件名对应已经绘制的图形对象，各图分别保存为可独立打开的页面。
chart_exports = {
    "plotly-bubble.html": bubble_fig,
    "plotly-pie.html": pie_fig,
    "plotly-donut.html": donut_fig,
    "plotly-treemap.html": treemap_fig,
    "plotly-sunburst.html": sunburst_fig,
    "plotly-sankey.html": sankey_fig,
}
# 保存路径列表供后续查看和清理，不重新构造图形或改动数据。
chart_paths = []
for filename, chart in chart_exports.items():
    chart_path = output_dir / filename
    chart.write_html(
        chart_path,
        include_plotlyjs=True,
        full_html=True,
        config=interaction_config,
        auto_open=False,
    )
    chart_paths.append(chart_path)
    print(filename, "：", chart_path.as_uri())
# 打开需要查看的页面，悬停读取数量，或点击地区进入下一层。

plotly-bubble.html ： file:///C:/Users/ZHUANG/AppData/Local/Temp/plotly-ch03-c_fkjjsq/plotly-bubble.html
plotly-pie.html ： file:///C:/Users/ZHUANG/AppData/Local/Temp/plotly-ch03-c_fkjjsq/plotly-pie.html


plotly-donut.html ： file:///C:/Users/ZHUANG/AppData/Local/Temp/plotly-ch03-c_fkjjsq/plotly-donut.html


plotly-treemap.html ： file:///C:/Users/ZHUANG/AppData/Local/Temp/plotly-ch03-c_fkjjsq/plotly-treemap.html
plotly-sunburst.html ： file:///C:/Users/ZHUANG/AppData/Local/Temp/plotly-ch03-c_fkjjsq/plotly-sunburst.html
plotly-sankey.html ： file:///C:/Users/ZHUANG/AppData/Local/Temp/plotly-ch03-c_fkjjsq/plotly-sankey.html


### 12.4 打开页面练习交互

打开第 12.2 节的目录，在浏览器中打开 plotly-inline.html。悬停 A4 应看到 4 min、27.0 °C、30 W；拖框放大后双击绘图区复原，单击 B 图例隐藏后再恢复。

再按兴趣打开其他图形：气泡图查看站点 C 的 80 单；饼图和环形图隐藏再恢复“线上”；矩形树图点击北区再用路径栏返回；旭日图查看北区设备占父级约 66.7%、占全部 40.0%；桑基图悬停“自动处理 → 已解决”，读取 50 件。

内嵌版可在无缓存、断网后重新加载；CDN 版仍需要取得外部脚本。仅在已打开的页面中操作，无法判断重新加载时是否还需网络。

完成查看后关闭标签页。需要分享时先保留文件，第 13 节提供清理方式。

## 13 选学：静态文件与清理

### 13.1 静态导出另有依赖

HTML 保留交互，PNG 或 SVG 用于静态展示。Figure.write_image 需要额外的 Kaleido；使用 Kaleido v1 时，Plotly.py 至少需要 6.1.1，并且系统中要有兼容的 Chrome 或 Chromium。Kaleido v1 不再自带 Chrome，本章的 Plotly 7.1 满足其 Python 包版本下限，但这不等于浏览器条件已经满足。

官方说明多数较新的 Chrome 或 Chromium 可用，并提供自动发现位置及 BROWSER_PATH 配置入口；没有给出一个适用于所有环境的固定浏览器版本号。因此应核对实际安装的 Kaleido、Plotly 和浏览器，再实际导出确认兼容性。主线依赖不包含 Kaleido，本章不自动安装包或下载浏览器。

以下入口默认不执行。具备上述条件后再设 export_static=True；width 和 height 是逻辑像素尺寸，PNG 的 scale=2 会把每个方向的像素数扩大两倍。导出后查看字体和裁切；SVG 也不保证所有图元都是矢量，例如 WebGL 系列可能包含栅格内容。

In [18]:
export_static = False
static_png = output_dir / "plotly-report.png"
static_svg = output_dir / "plotly-report.svg"

if export_static:
    report_fig.write_image(static_png, width=900, height=480, scale=2)
    report_fig.write_image(static_svg, width=900, height=480)
    print("静态文件已写入：", static_png, static_svg)
    # PNG 应为 1800×960 像素；仍须实际打开检查文字、图例和边界。
else:
    print("未执行选学静态导出；主线 HTML 不需要 Kaleido。")  # 预期：默认 export_static=False，只显示跳过静态导出的提示。

未执行选学静态导出；主线 HTML 不需要 Kaleido。


### 13.2 查看后清理导出文件

完成查看后，将 cleanup 改为 True 再运行。这里只删除本次使用的八个 HTML 和可选静态文件，再删除空目录；目录中若另存了其他文件，rmdir 会拒绝删除非空目录。Notebook 内保存的图形定义不受这些临时文件删除影响。

In [19]:
cleanup = False
if cleanup:
    for path in [inline_path, cdn_path, static_png, static_svg] + chart_paths:
        path.unlink(missing_ok=True)
    output_dir.rmdir()
    print("已清理本次导出文件。")  # 预期：仅在 cleanup=True 且删除完成后显示。
else:
    print("保留文件，请先完成浏览器查看。")  # 预期：默认 cleanup=False，显示此提示并保留 HTML。

保留文件，请先完成浏览器查看。


## 本章小结

（1）Plotly Express 把表格字段映射为位置、颜色和点形，返回可继续修改的 Figure。先明确每行观测与组别，再选择图形。

（2）data 保存系列，layout 保存整体设置；update_traces 与 update_layout 分工不同，config 还需在显示和导出时分别传入。

（3）hover_data 适合简单选字段与格式化，hovertemplate 适合明确的字段顺序和单位。悬停能补充信息，不能代替图上基本标签。

（4）缩放、平移和图例切换改变当前视图，不改变原始观测。图形导出后仍需重新打开并操作。

（5）气泡图把数量映射到面积；饼图与环形图表达同一整体的组成。读取精确数量仍应核对悬停，比较占比先确认分母与显示类别。

（6）矩形树图与旭日图表达父子组成，父级总量已经包含子级；桑基图表达不同阶段之间的流量，同一对象可能经过多条连接。

（7）内嵌 HTML 与 CDN HTML 的差别在于脚本资源是否随文件提供；静态导出又有 Kaleido 和浏览器条件。

自查：能否根据关系、占比、层级或流向选择图形，说明百分比的分母与重复计数风险，并判断关闭 Python 后哪些 HTML 仍具备离线加载条件？

## 练习

（1）为独立样本选择图形

下面每行是一块独立样品的模拟质量与强度，sample 只是编号，没有时间顺序。选择图形展示两变量关系，用悬停显示样品编号，并说明连成折线会额外暗示什么。

可验证标准：保留 4 个样本，质量与强度轴分别标 g 与 MPa；C3 的提示包含 14 g 和 40 MPa；说明所选图形突出关系、但不直接证明因果。

In [20]:
exercise_samples = pd.DataFrame(
    {
        "sample": ["C1", "C2", "C3", "C4"],
        "mass_g": [10, 12, 14, 16],
        "strength_mpa": [28, 35, 40, 42],
    }
)
# 在这里选择图形，设置字段标签和悬停编号，再调用 show()。
# 用 print() 说明图形突出或隐藏的信息。

（2）改变悬停方式并核对字段

用 measurements 重新创建图，把提示改为“方案、时间、温度、记录编号”，温度保留两位小数。分别使用 closest 与 x unified，解释逐点查看和同一时刻比较的区别。

可验证标准：A4 显示 27.00 °C，B4 显示 24.00 °C；编号未与另一组错配；两组同一时刻只有一个读数时，统一提示能比较两组值。

In [21]:
exercise_data = measurements.copy()
# 在这里设置 custom_data 和 hovertemplate，注意索引顺序。
# 创建两种 hovermode 的图，逐项悬停并核对读数。

（3）为离线分享选择导出方式

把上一题的图导出为一份可独立打开的 HTML，复用缩放和图例配置。在浏览器禁用缓存并断网后重新加载，核对悬停、缩放、复原与图例切换；查看完成后再清理本题文件。

可验证标准：解释选择 include_plotlyjs 值的原因；说明重新加载与只操作已打开页面的区别；尝试隐藏并恢复图例。

In [22]:
# 在这里新建临时目录并导出 HTML。
# 在浏览器中重新打开，练习悬停、缩放和图例切换。
# 最后只清理自己在本题创建的文件。

（4）改变总量并选择层级图

把南区的配件订单从 30 单改为 50 单，使用矩形树图或旭日图重新呈现地区与产品组成。解释为什么不能把各层节点的值全部相加作为订单总量，以及该图突出什么、没有呈现什么。

可验证标准：全部 120 单、北区 60 单、南区 60 单；南区配件占南区约 83.3%，占全部约 41.7%。若使用显式父节点与 branchvalues="total"，同步更新父节点数值；悬停应与新数据一致。

In [23]:
exercise_orders = region_orders.copy()
exercise_orders.loc[
    (exercise_orders["region"] == "南区") & (exercise_orders["product"] == "配件"),
    "orders",
] = 50
# 选择层级图，先核对地区与整体的总量，再绘制并悬停读取。
# 用 print() 说明图形突出或隐藏的信息，以及百分比的两个分母。

## 练习提示与解析

以下对应练习（3）。先独立作答，卡住时依次查看提示，完成后再对照解析。

提示 1：区分图形数据是否内嵌与 Plotly.js 是否内嵌。

提示 2：离线检查必须重新加载页面；已经打开的页面可能仍在使用缓存或内存里的资源。

### 练习（3）参考解析

本题采用 include_plotlyjs=True，并保留完整 HTML、图形中的本地数据及原 config。这样文件同时携带图形规范与 Plotly.js；使用 "cdn" 则仍需取得外部运行库。若另外引用在线图片或数据，也需逐项处理，不能只看这个开关。

先在新浏览器上下文中阻断外部网络，再打开导出文件，依次检查悬停、缩放、复原和图例切换。合格记录应写清实际网络条件及观察结果；这里只给出检查方法，没有代替读者完成本题。文件保存成功、页面在线打开过，都不足以证明本次离线交付可用。

## 参考与引用来源

原有页面于 2026-09-21 核查，新增图形页面于 2026-09-22 核查。在线 API 参考页当前标注 7.0.0，本章在 Plotly.py 7.1.0 中核对所用接口并组织原创模拟示例；展示前端与静态导出的依赖条件分别说明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Plotly 官方文档 | 第 1～2 节：[Plotly Express 总览](https://plotly.com/python/plotly-express/)、[px.scatter 的字段映射参数](https://plotly.com/python-api-reference/generated/plotly.express.scatter.html)、[Discrete Color 的数据类型、颜色映射与顺序](https://plotly.com/python/discrete-color/)、[Displaying figures 的 plotly_mimetype 与默认 renderer](https://plotly.com/python/renderers/#plotly_mimetype)。第 3 节：[Figure Structure 的 data、layout、frames 与 trace](https://plotly.com/python/figure-structure/)、[Creating and Updating Figures 的 update_traces、selector 与 update_layout](https://plotly.com/python/creating-and-updating-figures/)。第 4～5 节：[Hover Text and Formatting 的 hover_data、hovertemplate、custom_data 与 unified hover](https://plotly.com/python/hover-text-and-formatting/)、[Line Charts 的 Data Order 与 markers](https://plotly.com/python/line-charts/)、[px.line](https://plotly.com/python-api-reference/generated/plotly.express.line.html)。第 6、12 节：[Configuration 的滚轮缩放、工具栏与 responsiveness](https://plotly.com/python/configuration-options/)、[Legends 的系列与图例行为](https://plotly.com/python/legend/)、[layout 的 dragmode、legend.itemclick 与 legend.itemdoubleclick](https://plotly.com/python/reference/layout/)、[Interactive HTML Export](https://plotly.com/python/interactive-html-export/)、[write_html 的 include_plotlyjs、full_html、config 与 auto_open](https://plotly.com/python-api-reference/generated/plotly.io.write_html.html)。第 7～11 节：[Bubble Charts 的 Express 与 Scaling the Size](https://plotly.com/python/bubble-charts/)、[Pie Charts 的 values/names、Repeated Labels、Donut Chart](https://plotly.com/python/pie-charts/)、[Pie 的 hole、hovertemplate 与 textinfo](https://plotly.com/python/reference/pie/)、[Treemap 的 rectangular DataFrame、pathbar 与 branchvalues](https://plotly.com/python/treemaps/)、[Treemap 的 hovertemplate](https://plotly.com/python/reference/treemap/#treemap-hovertemplate)、[Sunburst 的 Repeated Labels 与 Branchvalues](https://plotly.com/python/sunburst-charts/)、[Sunburst 的 ids、parents、hovertemplate 与 textinfo](https://plotly.com/python/reference/sunburst/)、[Sankey Diagram 的 Basic Sankey Diagram 与流量定义](https://plotly.com/python/sankey-diagram/)、[Sankey 的 node、link、valueformat 与 valuesuffix](https://plotly.com/python/reference/sankey/)。第 13 节：[Static Image Export 的 Kaleido、Chrome、width/height/scale 和 WebGL 边界](https://plotly.com/python/static-image-export/)。 |
| GitHub 上的 Plotly 官方项目 | 第 6 节：[Plotly.js 的 plot_config.js 中 doubleClick 配置](https://github.com/plotly/plotly.js/blob/main/src/plot_api/plot_config.js)。第 13 节：[Kaleido README 的 Migrating from v0 to v1](https://github.com/plotly/Kaleido#migrating-from-v0-to-v1)，Plotly.py 至少 6.1.1 与外部 Chrome 要求。 |
| Python 3.12 官方文档 | 第 12～13 节：[tempfile.mkdtemp](https://docs.python.org/3.12/library/tempfile.html#tempfile.mkdtemp)、[Path.stat](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.stat)、[PurePath.as_uri](https://docs.python.org/3.12/library/pathlib.html#pathlib.PurePath.as_uri)、[Path.unlink](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.unlink)、[Path.rmdir](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.rmdir)，临时目录、文件大小与清理。 |